# Run the pipeline  (PySpark)

This notebook doesn't do any analysis. It **runs the pipeline the way a scheduled job would** —
starting a Spark session, reading `config.yaml`, calling the code in `src/`, writing the
results, and checking they still match the analysis.

**Why it exists.** The analysis lives in `product_segmentation.ipynb`, where the thinking was
done in pandas. `src/` is the same logic, tidied into functions and moved onto **PySpark** so it
scales past one machine. Code nobody runs is just a nice-looking sketch — so this notebook runs
it end to end and checks it gives the same answer the analysis did. If the two ever disagree,
something has drifted and I want the job to shout, not publish quietly.

**Where the work runs.** Steps 1–2 are distributed in Spark, because their cost scales with the
number of *rows* (84,550 today, millions in a rollout). Steps 3–5 work on one row per product
(353 rows) — tiny at any realistic scale — so they collect to the driver and use scikit-learn,
which keeps the model *identical* to the notebook (Spark MLlib has no RobustScaler and a
different K-Means init, so it would change the answer for no gain). See `docs/PRODUCTION.md`.

| | | |
|---|---|---|
| 1 | `clean.py`         | *(Spark)*  drop placeholder rows, add barcodes together, set aside new products |
| 2 | `features.py`      | *(Spark)*  one row per product: size, price, availability, dynamics, promo response |
| 3 | `group.py`         | *(driver)* prepare, choose how many groups, fit, name them, suggest a lever |
| 4 | `check.py`         | *(driver)* does it hold up? **publish, or stop and page someone** |
| 5 | `relationships.py` | *(Spark + driver)* which products compete, which sell together |

Nothing below has a number hard-coded in it. Every threshold comes from `config.yaml`.

In [1]:
## Optional — only needed if you're running this in Google Colab.
## Running locally? Skip this cell; the next one works out where it is by itself.
try:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/Product-Segmentation')
    !pip -q install pyspark
    print('Working directory:', os.getcwd())
except ImportError:
    print('Not in Colab — nothing to do here.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/Product-Segmentation


In [2]:
# Work from the project root, wherever this notebook is launched from.
import os, sys
from pathlib import Path

if not Path("config.yaml").exists() and Path("../config.yaml").exists():
    os.chdir("..")
sys.path.insert(0, ".")                       # so we can import from src/
print("Working from:", Path.cwd())

from pyspark.sql import SparkSession, functions as F
import pandas as pd

from src import config, clean, features, group, check, relationships

settings = config.load("config.yaml")

# A local session is enough for the recruitment dataset. On a cluster this is the only thing
# that changes — point master at the cluster and the four steps below are untouched.
spark = (SparkSession.builder
         .appName("product-segmentation")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "16")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("Spark", spark.version)
print(f"  drop rows below            £{settings.cleaning.minimum_price_per_pack} a pack")
print(f"  need at least              {settings.cleaning.minimum_weeks_of_history} weeks of history")
print(f"  search between             {settings.grouping.search_between[0]} and {settings.grouping.search_between[1]} groups")
print(f"  publish only if groups agree at  {settings.checks.minimum_agreement} or better")
print(f"  random seed                {settings.grouping.random_seed}")

Working from: /content/drive/MyDrive/Product-Segmentation
Spark 4.0.3
  drop rows below            £0.1 a pack
  need at least              13 weeks of history
  search between             3 and 10 groups
  publish only if groups agree at  0.6 or better
  random seed                42


---
## Step 1 — Data Cleaning  *(Spark)*
Read the raw sales file, drop the ~19% placeholder rows, add the barcodes of one product
together (sum sales, **max** availability), and set aside products with under 13 weeks of
history. This is the step that scales with rows, so it's the one that matters most in Spark.

In [3]:
raw = clean.load_raw(spark, settings.data.input)
sales, set_aside = clean.run(raw, settings)
sales = sales.cache()

print(f"in:   {raw.count():,} rows")
print(f"out:  {sales.count():,} rows, {sales.select('rgm_ppg').distinct().count()} products")
print(f"      {len(set_aside)} products set aside (not enough history)")

in:   84,550 rows
out:  63,834 rows, 353 products
      13 products set aside (not enough history)


---
## Step 2 — Feature Engineering  *(Spark)*
One row per product. Aggregates (how big, how expensive, how promoted) are a distributed
group-by; the per-product dynamics (growth, seasonality, promo-lift) run through
`applyInPandas` — one product to one core, the exact same numpy as the notebook. Blanks are
kept where the data can't answer (e.g. promo-lift needs enough promo *and* quiet weeks).

In [4]:
product_features = features.run(sales, settings).cache()
n = product_features.count()

feat_pdf = group.to_pandas(product_features)     # collect the tiny one-row-per-product table
blanks = feat_pdf[group.FEATURE_NAMES].isna().sum()
print(f"{n} products, {len(group.FEATURE_NAMES)} features each\n")
print("Blanks (deliberate — the data couldn't answer for these):")
print(blanks[blanks > 0].to_string())
print("\n  promo_lift blank = not enough promo weeks to compare against quiet ones")

353 products, 10 features each

Blanks (deliberate — the data couldn't answer for these):
promo_lift    35

  promo_lift blank = not enough promo weeks to compare against quiet ones


---
## Step 3 — Clustering & Grouping  *(driver / scikit-learn)*
Prepare (clip, fill blanks with the median, robust-scale, PCA to 90%), choose how many groups
(the search starts at 3 because K=2 puts 82% in one bucket), fit, name from readable rules, and
suggest one lever per group.

In [5]:
prepared, ready, fitted = group.prepare(feat_pdf, settings)
k, scores = group.choose_how_many(prepared, settings)

print(f"{len(group.FEATURE_NAMES)} features squashed into {prepared.shape[1]} directions\n")
print(scores.round(2).to_string(index=False))
print(f"\nNote k=2: it scores well, but puts {scores.loc[0,'biggest_group']:.0%} of products in ONE group.")
print(f"That's why the search starts at 3 (config.yaml). Chosen: {k} groups.")

10 features squashed into 5 directions

 groups  silhouette  second_score  biggest_group
      2        0.42        161.21           0.82
      3        0.41        194.12           0.66
      4        0.38        185.79           0.58
      5        0.30        170.18           0.42
      6        0.27        158.49           0.33
      7        0.27        154.73           0.33
      8        0.28        149.04           0.31
      9        0.28        140.93           0.33
     10        0.23        135.37           0.22

Note k=2: it scores well, but puts 82% of products in ONE group.
That's why the search starts at 3 (config.yaml). Chosen: 3 groups.


In [6]:
labels = group.fit(prepared, k, settings)
profile, names, revenue_pct = group.describe_groups(feat_pdf, labels)
suggestions = group.suggest(feat_pdf, labels, names, revenue_pct)
split = group.split_biggest(prepared, feat_pdf, labels, settings)

import pandas as pd
print(suggestions.to_string(index=False))
print("\nGroup sizes:", pd.Series(labels).value_counts().sort_index().tolist())
sub_sizes = split['sub'].value_counts().sort_values(ascending=False).tolist()
print(f"Biggest group ('{suggestions.iloc[0].group_name}') splits into {len(sub_sizes)} layers: {sub_sizes}")

     group_name  products  revenue_pct                      action                                     why
    Steady core       234         48.9            No single action     Too big and too mixed for one lever
  Promo leaders        58         41.4 Optimise the promo calendar Promo weeks clearly outsell quiet weeks
Shrinking shelf        61          9.7              Review product                  Investigate shelf loss

Group sizes: [234, 61, 58]
Biggest group ('Steady core') splits into 3 layers: [113, 85, 36]


---
## Step 4 — should we publish this?  *(driver)*
The step that can stop the pipeline. Re-run the grouping on 25 random 80% samples; if the groups
don't hold up, the new ones are **not** published — the old ones stay live and someone gets paged.

In [7]:
agreement = check.does_it_hold_up(lambda X: group.fit(X, k, settings), prepared, settings)
publish, reason = check.should_we_publish(agreement, suggestions, settings)

print(f"Re-ran the grouping on {settings.checks.resample_times} random "
      f"{settings.checks.resample_size:.0%} samples.")
print(f"Agreement with the full answer: {agreement:.2f}\n")
print(f"Publish? {reason}")

# Compare with the previous run, if there was one.
previous_path = Path("output/groups.csv")
if previous_path.exists():
    previous = pd.read_csv(previous_path)[["rgm_ppg", "group"]]
    current = feat_pdf.assign(group=labels)[["rgm_ppg", "group"]]
    moved = check.how_many_products_moved(previous, current)
    print(f"\nSince the last run, {moved:.0%} of products changed group.")
    if moved > 0.20:
        print("That's a lot. I'd redraw the map and TELL people, not quietly republish.")

Re-ran the grouping on 25 random 80% samples.
Agreement with the full answer: 0.94

Publish? Yes. Groups hold up at 0.94. 2 of 3 groups get a lever; 1 is too big and mixed for one and gets split instead.


---
## Step 5 — which products compete, which sell together  *(Spark + driver)*
The weekly aggregation is distributed; the pairwise cross-price fits run on the top-40 sellers
of the biggest subcategory (O(n²), so it's capped). A link survives only if it clears a
Bonferroni cut and an effect-size floor.

In [8]:
links = relationships.find_links(sales, settings)
print(f"{len(links)} strong links: "
      f"{int((links.effect > 0).sum())} compete, {int((links.effect < 0).sum())} sell together")
print(links.head(6).to_string(index=False))

45 strong links: 28 compete, 17 sell together
                                                                                                                                                                                                       product_a                                                                                                                                                                                                        product_b   effect            p relationship
PPG_ATTR1_F6394E71 | PPG_ATTR2_2BEF0FAC | PPG_ATTR3_E06D3901 | PPG_ATTR4_AE186179 | PPG_ATTR5_368A0D73 | PPG_ATTR6_793F866E | PPG_ATTR7_A87D5006 | PPG_ATTR8_A718BD30 | PPG_ATTR9_C4CA4238 | PPG_ATTR12_2BEF0FAC PPG_ATTR1_F6394E71 | PPG_ATTR2_2BEF0FAC | PPG_ATTR3_E06D3901 | PPG_ATTR4_AE186179 | PPG_ATTR5_368A0D73 | PPG_ATTR6_793F866E | PPG_ATTR7_A87D5006 | PPG_ATTR8_7FE34FAD | PPG_ATTR9_C4CA4238 | PPG_ATTR12_2BEF0FAC 0.487048 5.107026e-15      compete
PPG_ATTR1_F6394E71 | PPG_ATTR2_2BEF0FAC | PPG_AT

---
## Publish
Write the results. `output/` is the business-facing set; the analysis notebook writes a few
richer exploratory files alongside (see `output/README.md`).

In [9]:
if publish:
    out = Path("output"); out.mkdir(exist_ok=True)

    groups_table = feat_pdf.assign(group=labels)
    groups_table["group_name"] = groups_table.group.map(names)
    groups_table.to_csv(out / "groups.csv", index=False)

    profile.rename(index=names).rename_axis("group_name").to_csv(out / "group_profiles.csv")
    suggestions.to_csv(out / "suggestions.csv", index=False)
    pd.Series(set_aside, name="rgm_ppg").to_csv(out / "products_set_aside.csv", index=False)
    split.to_csv(out / "biggest_group_split.csv", index=False)
    links.to_csv(out / "cross_price_links.csv", index=False)

    print("Published. Written to output/:")
    for f in ["groups.csv", "group_profiles.csv", "suggestions.csv",
              "products_set_aside.csv", "biggest_group_split.csv", "cross_price_links.csv"]:
        print("  ", f)
else:
    print("NOT published. The previous groups stay live.")
    print("(In production this is where the job exits non-zero and pages whoever is on call.)")

Published. Written to output/:
   groups.csv
   group_profiles.csv
   suggestions.csv
   products_set_aside.csv
   biggest_group_split.csv
   cross_price_links.csv


---
## Does this match the analysis notebook?
The whole point of `src/` is that it's the same logic on a different engine. So it had better
give the same answer. If these disagree, someone changed one and not the other.

In [10]:
expected = {
    "products": 353,
    "groups": 3,
    "group_sizes": [58, 61, 234],
    "group_names": ["Promo leaders", "Shrinking shelf", "Steady core"],
}
got = {
    "products": len(feat_pdf),
    "groups": k,
    "group_sizes": sorted(pd.Series(labels).value_counts().tolist()),
    "group_names": sorted(names.values()),
}
for what, should_be in expected.items():
    tick = "OK  " if got[what] == should_be else "DIFFERENT"
    print(f"{tick}  {what:<12} analysis says {should_be},  pipeline says {got[what]}")

print()
if got == expected:
    print("The pipeline reproduces the analysis exactly. The PySpark code in src/ isn't a")
    print("sketch — it's the same thing, and it runs at scale.")
else:
    print("They DISAGREE. Either src/ or the analysis notebook changed and the other hasn't")
    print("caught up. Worth fixing before anyone relies on this.")

spark.stop()

OK    products     analysis says 353,  pipeline says 353
OK    groups       analysis says 3,  pipeline says 3
OK    group_sizes  analysis says [58, 61, 234],  pipeline says [58, 61, 234]
OK    group_names  analysis says ['Promo leaders', 'Shrinking shelf', 'Steady core'],  pipeline says ['Promo leaders', 'Shrinking shelf', 'Steady core']

The pipeline reproduces the analysis exactly. The PySpark code in src/ isn't a
sketch — it's the same thing, and it runs at scale.
